# comparing model performance acrossed balanced datasets

In [52]:
# Setup and Create Balanced YBT Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, f1_score
import joblib
import os

print("="*50)
print("CREATING BALANCED YBT DATASET")
print("="*50)

# Load YBT data
ybt_df = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_processed.csv')
print(f"Original YBT shape: {ybt_df.shape}")
print(f"YBT class distribution: {ybt_df['autism_target'].value_counts()}")

# Create balanced YBT dataset
autism_cases = ybt_df[ybt_df['autism_target'] == 1]
control_cases = ybt_df[ybt_df['autism_target'] == 0]

print(f"Autism cases: {len(autism_cases)}")
print(f"Control cases: {len(control_cases)}")

# Sample equal numbers
min_class_size = min(len(autism_cases), len(control_cases))
print(f"Balancing to {min_class_size} cases per class")

# Sample balanced dataset
autism_balanced = autism_cases.sample(n=min_class_size, random_state=42)
control_balanced = control_cases.sample(n=min_class_size, random_state=42)

# Combine balanced datasets
ybt_balanced = pd.concat([autism_balanced, control_balanced], ignore_index=True)
ybt_balanced = ybt_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

print(f"Balanced YBT shape: {ybt_balanced.shape}")
print(f"Balanced YBT class distribution: {ybt_balanced['autism_target'].value_counts()}")

# Save balanced YBT dataset
ybt_balanced.to_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced.csv', index=False)
print("Saved balanced YBT dataset")

CREATING BALANCED YBT DATASET
Original YBT shape: (19479, 122)
YBT class distribution: autism_target
0    18732
1      747
Name: count, dtype: int64
Autism cases: 747
Control cases: 18732
Balancing to 747 cases per class
Balanced YBT shape: (1494, 122)
Balanced YBT class distribution: autism_target
0    747
1    747
Name: count, dtype: int64
Saved balanced YBT dataset


/var/folders/1b/r6y9_1zx175chs6rm5vj_xfc0000gp/T/ipykernel_17204/1977690999.py:15: DtypeWarning: Columns (15,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  ybt_df = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_processed.csv')


In [53]:
# 2. Standardize YBT Data to Match C4 Scale
print("="*50)
print("STANDARDIZING YBT DATA TO MATCH C4 SCALE")
print("="*50)

# Load datasets
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced.csv')

print(f"Original C4 shape: {c4_balanced.shape}")
print(f"Original YBT shape: {ybt_balanced.shape}")

# Create standardized YBT dataset
ybt_standardized = ybt_balanced.copy()

# Standardize questionnaire items to match C4 distribution
for i in range(1, 11):
    eq_col = f'eq_{i}'
    sqr_col = f'sqr_{i}'
    aq_col = f'aq_{i}'
    
    if eq_col in ybt_standardized.columns and eq_col in c4_balanced.columns:
        # Map YBT 1-4 scale to C4 scale
        ybt_min = ybt_standardized[eq_col].min()
        ybt_max = ybt_standardized[eq_col].max()
        ybt_normalized = (ybt_standardized[eq_col] - ybt_min) / (ybt_max - ybt_min)
        
        c4_min = c4_balanced[eq_col].min()
        c4_max = c4_balanced[eq_col].max()
        ybt_standardized[eq_col] = ybt_normalized * (c4_max - c4_min) + c4_min
    
    if sqr_col in ybt_standardized.columns and sqr_col in c4_balanced.columns:
        ybt_min = ybt_standardized[sqr_col].min()
        ybt_max = ybt_standardized[sqr_col].max()
        ybt_normalized = (ybt_standardized[sqr_col] - ybt_min) / (ybt_max - ybt_min)
        
        c4_min = c4_balanced[sqr_col].min()
        c4_max = c4_balanced[sqr_col].max()
        ybt_standardized[sqr_col] = ybt_normalized * (c4_max - c4_min) + c4_min
    
    if aq_col in ybt_standardized.columns and aq_col in c4_balanced.columns:
        ybt_min = ybt_standardized[aq_col].min()
        ybt_max = ybt_standardized[aq_col].max()
        ybt_normalized = (ybt_standardized[aq_col] - ybt_min) / (ybt_max - ybt_min)
        
        c4_min = c4_balanced[aq_col].min()
        c4_max = c4_balanced[aq_col].max()
        ybt_standardized[aq_col] = ybt_normalized * (c4_max - c4_min) + c4_min

# Standardize totals and age
for col in ['eq_total', 'sqr_total', 'aq_total', 'age']:
    if col in ybt_standardized.columns and col in c4_balanced.columns:
        ybt_min = ybt_standardized[col].min()
        ybt_max = ybt_standardized[col].max()
        ybt_normalized = (ybt_standardized[col] - ybt_min) / (ybt_max - ybt_min)
        
        c4_min = c4_balanced[col].min()
        c4_max = c4_balanced[col].max()
        ybt_standardized[col] = ybt_normalized * (c4_max - c4_min) + c4_min

print("\nAfter standardization (first 5 EQ items):")
for i in range(1, 6):
    col = f'eq_{i}'
    if col in ybt_standardized.columns:
        print(f"  {col}: mean={ybt_standardized[col].mean():.3f}, std={ybt_standardized[col].std():.3f}")

# Add feature engineered items using standardized data
# 1. age_x_eq
ybt_standardized['age_x_eq'] = ybt_standardized['age'] * ybt_standardized['eq_total']

# 2. aq_eq_interaction
ybt_standardized['aq_eq_interaction'] = ybt_standardized['aq_total'] * ybt_standardized['eq_total']

# 3. eq_sqr_ratio
ybt_standardized['eq_sqr_ratio'] = ybt_standardized['eq_total'] / (ybt_standardized['sqr_total'] + 1e-8)

# 4. log_aq_total
ybt_standardized['log_aq_total'] = np.log1p(ybt_standardized['aq_total'])

# 5. sqrt_age
ybt_standardized['sqrt_age'] = np.sqrt(ybt_standardized['age'])

# 6. high_aq
aq_mean = ybt_standardized['aq_total'].mean()
aq_std = ybt_standardized['aq_total'].std()
ybt_standardized['high_aq'] = (ybt_standardized['aq_total'] > aq_mean + aq_std).astype(int)

# Save standardized YBT dataset
ybt_standardized.to_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv', index=False)
print(f"\nSaved standardized YBT dataset: {ybt_standardized.shape}")

# Verify standardization worked
print("\nVerification - comparing C4 vs YBT means:")
for i in range(1, 6):
    col = f'eq_{i}'
    if col in ybt_standardized.columns and col in c4_balanced.columns:
        c4_mean = c4_balanced[col].mean()
        ybt_mean = ybt_standardized[col].mean()
        print(f"  {col}: C4={c4_mean:.3f}, YBT={ybt_mean:.3f}, diff={abs(c4_mean-ybt_mean):.3f}")

STANDARDIZING YBT DATA TO MATCH C4 SCALE
Original C4 shape: (90538, 55)
Original YBT shape: (1494, 122)

After standardization (first 5 EQ items):
  eq_1: mean=1.285, std=1.047
  eq_2: mean=0.943, std=1.002
  eq_3: mean=-0.394, std=0.955
  eq_4: mean=0.566, std=1.039
  eq_5: mean=-0.167, std=0.933

Saved standardized YBT dataset: (1494, 122)

Verification - comparing C4 vs YBT means:
  eq_1: C4=0.215, YBT=1.285, diff=1.070
  eq_2: C4=0.162, YBT=0.943, diff=0.780
  eq_3: C4=-0.212, YBT=-0.394, diff=0.182
  eq_4: C4=0.172, YBT=0.566, diff=0.395
  eq_5: C4=-0.186, YBT=-0.167, diff=0.019


/Users/eb2007/playground/bullpy/c4_play2/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [54]:
# 3. Add Missing Features to C4 Dataset
print("="*50)
print("ADDING MISSING FEATURE ENGINEERED ITEMS TO C4")
print("="*50)

# Load the C4 balanced dataset
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')

print(f"Original C4 balanced shape: {c4_balanced.shape}")

# Add missing feature engineered items to C4
# 1. age_x_eq (age × EQ interaction)
if 'age_x_eq' not in c4_balanced.columns:
    c4_balanced['age_x_eq'] = c4_balanced['age'] * c4_balanced['eq_total']
    print("✓ Added age_x_eq to C4")

# 2. aq_eq_interaction (AQ × EQ interaction)
if 'aq_eq_interaction' not in c4_balanced.columns:
    c4_balanced['aq_eq_interaction'] = c4_balanced['aq_total'] * c4_balanced['eq_total']
    print("✓ Added aq_eq_interaction to C4")

# 3. eq_sqr_ratio (EQ/SQR ratio)
if 'eq_sqr_ratio' not in c4_balanced.columns:
    c4_balanced['eq_sqr_ratio'] = c4_balanced['eq_total'] / (c4_balanced['sqr_total'] + 1e-8)
    print("✓ Added eq_sqr_ratio to C4")

# 4. log_aq_total (log transformation of AQ total)
if 'log_aq_total' not in c4_balanced.columns:
    c4_balanced['log_aq_total'] = np.log1p(c4_balanced['aq_total'])
    print("✓ Added log_aq_total to C4")

# 5. sqrt_age (square root transformation of age)
if 'sqrt_age' not in c4_balanced.columns:
    c4_balanced['sqrt_age'] = np.sqrt(c4_balanced['age'])
    print("✓ Added sqrt_age to C4")

# 6. high_aq (boolean high AQ indicator)
if 'high_aq' not in c4_balanced.columns:
    aq_mean = c4_balanced['aq_total'].mean()
    aq_std = c4_balanced['aq_total'].std()
    c4_balanced['high_aq'] = (c4_balanced['aq_total'] > aq_mean + aq_std).astype(int)
    print("✓ Added high_aq to C4")

print(f"\nUpdated C4 balanced shape: {c4_balanced.shape}")

# Save the updated C4 dataset
c4_balanced.to_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced_enhanced.csv', index=False)
print("Saved enhanced C4 balanced dataset")

ADDING MISSING FEATURE ENGINEERED ITEMS TO C4
Original C4 balanced shape: (90538, 55)
✓ Added age_x_eq to C4
✓ Added aq_eq_interaction to C4
✓ Added eq_sqr_ratio to C4
✓ Added log_aq_total to C4
✓ Added sqrt_age to C4
✓ Added high_aq to C4

Updated C4 balanced shape: (90538, 61)


/Users/eb2007/playground/bullpy/c4_play2/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Saved enhanced C4 balanced dataset


In [55]:
# 4. Load and Compare Standardized Datasets
print("="*50)
print("LOADING AND COMPARING STANDARDIZED DATASETS")
print("="*50)

# Load enhanced datasets
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced_enhanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

print(f"C4 enhanced balanced shape: {c4_balanced.shape}")
print(f"YBT enhanced balanced shape: {ybt_balanced.shape}")

# Check feature alignment
exclude_cols = ['autism_target', 'userid']
c4_features = [col for col in c4_balanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_balanced.columns if col not in exclude_cols]

print(f"\nC4 features: {len(c4_features)}")
print(f"YBT features: {len(ybt_features)}")

# Find common features (excluding SPQ features)
spq_features = [col for col in c4_features if col.startswith('spq_')]
common_features = [col for col in c4_features if col in ybt_features and col not in spq_features]

print(f"\nSPQ features in C4: {len(spq_features)}")
print(f"Common features (excluding SPQ): {len(common_features)}")

# Check for key features
key_features_to_check = [
    'age_x_aq', 'age_x_eq', 'aq_eq_interaction', 'eq_sqr_ratio', 
    'log_aq_total', 'sqrt_age', 'high_aq', 'd_score'
]

print(f"\nChecking for key feature engineered items:")
for feature in key_features_to_check:
    if feature in common_features:
        print(f"  ✓ {feature}")
    else:
        print(f"  ✗ {feature} (missing)")

print(f"\nFinal aligned features: {len(common_features)}")

# Verify standardization worked
print(f"\nVerifying standardization (first 5 features):")
for feature in common_features[:5]:
    c4_mean = c4_balanced[feature].mean()
    ybt_mean = ybt_balanced[feature].mean()
    print(f"  {feature}: C4 mean={c4_mean:.3f}, YBT mean={ybt_mean:.3f}, diff={abs(c4_mean-ybt_mean):.3f}")

LOADING AND COMPARING STANDARDIZED DATASETS
C4 enhanced balanced shape: (90538, 61)
YBT enhanced balanced shape: (1494, 122)

C4 features: 60
YBT features: 121

SPQ features in C4: 11
Common features (excluding SPQ): 44

Checking for key feature engineered items:
  ✓ age_x_aq
  ✓ age_x_eq
  ✓ aq_eq_interaction
  ✓ eq_sqr_ratio
  ✓ log_aq_total
  ✓ sqrt_age
  ✓ high_aq
  ✓ d_score

Final aligned features: 44

Verifying standardization (first 5 features):
  age: C4 mean=26.003, YBT mean=13.359, diff=12.644
  eq_1: C4 mean=0.215, YBT mean=1.285, diff=1.070
  eq_2: C4 mean=0.162, YBT mean=0.943, diff=0.780
  eq_3: C4 mean=-0.212, YBT mean=-0.394, diff=0.182
  eq_4: C4 mean=0.172, YBT mean=0.566, diff=0.395


In [56]:
#5: Train Model on C4 Balanced Data
print("="*50)
print("TRAINING MODEL ON C4 BALANCED DATA")
print("="*50)

# Prepare C4 data with shared features only
X_c4 = c4_balanced[common_features]
y_c4 = c4_balanced['autism_target']

print(f"C4 training data shape: {X_c4.shape}")
print(f"C4 class distribution: {y_c4.value_counts()}")

# Split C4 data
X_train, X_test, y_train, y_test = train_test_split(
    X_c4, y_c4, test_size=0.2, stratify=y_c4, random_state=42
)

# Train model
rf_c4 = RandomForestClassifier(
    n_estimators=200, 
    random_state=42, 
    class_weight='balanced',
    max_depth=15
)

rf_c4.fit(X_train, y_train)

# Evaluate on C4 test set
y_probs_c4 = rf_c4.predict_proba(X_test)[:, 1]
y_pred_c4 = rf_c4.predict(X_test)

print("\nC4 Test Set Performance:")
print(classification_report(y_test, y_pred_c4))
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs_c4):.3f}")

# Threshold tuning on C4
prec, rec, thresholds = precision_recall_curve(y_test, y_probs_c4)
f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thresh_c4 = thresholds[np.argmax(f1s)]

print(f"\nBest threshold for C4: {best_thresh_c4:.3f}")

y_pred_c4_optimal = (y_probs_c4 >= best_thresh_c4).astype(int)
print(f"F1 at optimal threshold: {f1_score(y_test, y_pred_c4_optimal):.3f}")

# Save model
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/models', exist_ok=True)
joblib.dump(rf_c4, '/Users/eb2007/playground/bullpy/c4_play2/models/rf_c4_balanced.joblib')

print("\nModel saved successfully")

TRAINING MODEL ON C4 BALANCED DATA
C4 training data shape: (90538, 44)
C4 class distribution: autism_target
0    45269
1    45269
Name: count, dtype: int64

C4 Test Set Performance:
              precision    recall  f1-score   support

           0       0.68      0.69      0.69      9054
           1       0.69      0.68      0.68      9054

    accuracy                           0.68     18108
   macro avg       0.68      0.68      0.68     18108
weighted avg       0.68      0.68      0.68     18108

ROC-AUC: 0.758

Best threshold for C4: 0.393
F1 at optimal threshold: 0.718

Model saved successfully


In [57]:
# 6. Test Model on YBT Balanced Data
print("="*50)
print("TESTING C4 MODEL ON YBT BALANCED DATA")
print("="*50)

# Prepare YBT data with shared features
X_ybt = ybt_balanced[common_features]
y_ybt = ybt_balanced['autism_target']

print(f"YBT test data shape: {X_ybt.shape}")
print(f"YBT class distribution: {y_ybt.value_counts()}")

# Get predictions from C4 model
y_probs_ybt = rf_c4.predict_proba(X_ybt)[:, 1]
y_pred_ybt = rf_c4.predict(X_ybt)

print("\nYBT Performance with C4 Model:")
print(classification_report(y_ybt, y_pred_ybt))
print(f"ROC-AUC: {roc_auc_score(y_ybt, y_probs_ybt):.3f}")

# Apply C4 optimal threshold
y_pred_ybt_c4_threshold = (y_probs_ybt >= best_thresh_c4).astype(int)
print(f"\nYBT Performance with C4 threshold ({best_thresh_c4:.3f}):")
print(classification_report(y_ybt, y_pred_ybt_c4_threshold))
print(f"F1 score: {f1_score(y_ybt, y_pred_ybt_c4_threshold):.3f}")

# Find optimal threshold for YBT
prec_ybt, rec_ybt, thresholds_ybt = precision_recall_curve(y_ybt, y_probs_ybt)
f1s_ybt = 2 * (prec_ybt * rec_ybt) / (prec_ybt + rec_ybt + 1e-8)
best_thresh_ybt = thresholds_ybt[np.argmax(f1s_ybt)]

print(f"\nBest threshold for YBT: {best_thresh_ybt:.3f}")

y_pred_ybt_optimal = (y_probs_ybt >= best_thresh_ybt).astype(int)
print(f"YBT Performance with YBT optimal threshold:")
print(classification_report(y_ybt, y_pred_ybt_optimal))
print(f"F1 score: {f1_score(y_ybt, y_pred_ybt_optimal):.3f}")

TESTING C4 MODEL ON YBT BALANCED DATA
YBT test data shape: (1494, 44)
YBT class distribution: autism_target
0    747
1    747
Name: count, dtype: int64

YBT Performance with C4 Model:
              precision    recall  f1-score   support

           0       0.31      0.14      0.19       747
           1       0.44      0.69      0.54       747

    accuracy                           0.41      1494
   macro avg       0.38      0.41      0.36      1494
weighted avg       0.38      0.41      0.36      1494

ROC-AUC: 0.325

YBT Performance with C4 threshold (0.393):
              precision    recall  f1-score   support

           0       0.19      0.03      0.05       747
           1       0.48      0.88      0.62       747

    accuracy                           0.46      1494
   macro avg       0.34      0.46      0.33      1494
weighted avg       0.34      0.46      0.33      1494

F1 score: 0.619

Best threshold for YBT: 0.159
YBT Performance with YBT optimal threshold:
            

/Users/eb2007/playground/bullpy/c4_play2/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/eb2007/playground/bullpy/c4_play2/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/eb2007/playground/bullpy/c4_play2/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera